# Microsoft Agent Framework & Agent-to-Agent (A2A) Protocol

This notebook demonstrates how to implement Agent-to-Agent (A2A) communication using Microsoft Agent Framework. We'll create two agents:

1. **Travel Booking Agent**: A travel planning assistant that helps users with travel arrangements
2. **Flight Booking Agent**: A specialized agent that handles flight bookings through A2A communication

## What You'll Learn

- How to create agents using Microsoft Agent Framework
- How to implement A2A communication between agents
- How to set up tool functions for agent collaboration
- How to run A2A agents in production using FastAPI

## Prerequisites

- Azure OpenAI or OpenAI API access
- Python environment with Microsoft Agent Framework installed
- Understanding of async/await patterns in Python

Let's get started!

## Prerequisites and Setup

First, let's install the required dependencies and set up our environment.


In [ ]:
# Install required packages
# If you already did the uv configuration, you can skip this step
#%pip install semantic-kernel python-dotenv fastapi uvicorn httpx a2a-sdk

In [ ]:
import os
import asyncio
import json
from uuid import uuid4
from typing import Annotated

from dotenv import load_dotenv
from pydantic import Field

# Microsoft Agent Framework imports
from agent_framework import ChatAgent
from agent_framework.azure import AzureChatClient
from azure.identity import AzureCliCredential

# A2A Protocol imports
from a2a.client import A2ACardResolver, A2AClient
from a2a.types import MessageSendParams, SendMessageRequest

# Load environment variables
load_dotenv('.env')

# Verify environment setup
print("Environment setup:")
print(f"✓ Azure CLI authentication available")
print(f"✓ Microsoft Agent Framework imported successfully")
print(f"✓ A2A Protocol components imported successfully")

## Building the Flight Booking Agent

Let's start by creating our Flight Booking Agent using Microsoft Agent Framework. This agent will:

1. **Microsoft Agent Framework Integration**: We use Microsoft Agent Framework's `AzureChatClient` to create our Agent
2. **Azure Authentication**: Leverage Azure CLI credentials for seamless authentication  
3. **Thread Management**: Automatic conversation thread handling for stateful interactions
4. **A2A Integration**: Expose the agent through A2A protocol for inter-agent communication

### Key Features:
- **Automatic Thread Management**: No manual thread handling required
- **Direct Client Usage**: No kernel or complex setup needed
- **Built-in Azure Integration**: Native Azure OpenAI support
- **Simplified API**: Single `run()` method for all interactions

Let's implement this step by step:

In [ ]:
class MicrosoftAgentFrameworkFlightBookingAgent:
    """A flight booking agent using Microsoft Agent Framework and Azure OpenAI."""
    
    def __init__(self):
        """Initialize the flight booking agent with Azure OpenAI service."""
        print("Initializing Microsoft Agent Framework Flight Booking Agent...")
        
        # Create Azure client with automatic authentication
        client = AzureChatClient(credential=AzureCliCredential())
        
        # Create the agent with instructions
        self.chat_agent = client.create_agent(
            instructions=(
                "You are a helpful flight booking assistant. "
                "Your task is to help users book flights by gathering necessary information "
                "such as departure city, destination city, travel dates, number of passengers, "
                "and preferred class of service. Once you have all the required information, "
                "provide a confirmation summary and simulate a successful booking."
            )
        )
        
        # Store chat threads per context to maintain conversation state
        self.thread_store: dict[str, object] = {}
        
        print("✓ Microsoft Agent Framework Flight Booking Agent initialized successfully!")
    
    def _get_or_create_thread(self, context_id: str) -> object:
        """Get existing thread or create a new one for the given context."""
        thread = self.thread_store.get(context_id)
        
        if thread is None:
            # Agent Framework handles thread creation automatically
            thread = self.chat_agent.get_new_thread()
            self.thread_store[context_id] = thread
            print(f"✓ Created new thread for context ID: {context_id}")
        
        return thread
    
    async def book_flight(self, user_input: str, context_id: str) -> str:
        """
        Process a flight booking request from the user.
        
        Args:
            user_input: The user's request for flight booking
            context_id: The context ID for maintaining conversation state
            
        Returns:
            The response from the flight booking agent
            
        Raises:
            ValueError: If user input is empty
        """
        print(f"Processing flight booking request: {user_input} (Context: {context_id})")
        
        if not user_input or not user_input.strip():
            raise ValueError("User input cannot be empty.")
        
        try:
            # Get or create thread for the context  
            thread = self._get_or_create_thread(context_id)
            
            # Use the unified run API - much simpler than before!
            response = await self.chat_agent.run(user_input, thread=thread)
            
            print(f"✓ Flight booking agent response generated")
            
            return response.text
            
        except Exception as e:
            print(f"✗ Error processing flight booking request: {e}")
            return f"I apologize, but I encountered an error while processing your flight booking request: {str(e)}"

# Test the agent
flight_agent = MicrosoftAgentFrameworkFlightBookingAgent()
print("Flight booking agent ready for testing!")

### Test the Flight Booking Agent

Let's test our flight booking agent directly to see how it works:


In [ ]:
# Test the flight booking agent directly
test_response = await flight_agent.book_flight(
    "I need to book a flight from Seattle to New York for next Friday", 
    "test_context_1"
)
print("Agent Response:")
print(test_response)

### What Just Happened?

In the test above, notice how the agent:

1. **Extracted Information**: It identified the departure city (New York), destination (London), and date (July 15th) from the user's natural language input
2. **Identified Missing Information**: It recognized that it still needed the number of passengers and class preference
3. **Maintained Context**: It structured the response in a clear, organized way
4. **Guided the Conversation**: It asked for the remaining information in a user-friendly manner

This demonstrates the agent's ability to understand partial information and guide users through a complete booking process.


### Completing the Booking Flow

Let's continue the conversation by providing the missing information:


In [ ]:
response = await flight_agent.book_flight(
    user_input="I need it for 2 passengers, business class, mock the rest",
    context_id="test_context_1"
)
print("-" * 50)
print("Final Response")
print("-" * 50)
print(f"Agent: {response}")
print("-" * 50)

### Conversation Continuity in Action

Perfect! The agent:

1. **Remembered the Context**: It recalled all the previous information from our conversation
2. **Processed New Information**: It understood "2 passengers" and "business class"
3. **Provided Confirmation**: It summarized all the booking details
4. **Simulated Success**: It confirmed the booking as if it were a real transaction

This shows how the conversation history is maintained within the same context ID (`test_context_1`).


## Creating the Travel Agent with Tool Functions

Now let's create the Travel Agent that will communicate with our Flight Booking Agent using A2A protocol. In Microsoft Agent Framework, tool registration is much simpler - we just use regular Python functions with type annotations!

In [ ]:
response = await flight_agent.book_flight(
        "Book a flight to Paris tomorrow", 
        "test_context_2"
    )
print("-" * 50)
print("Final Response")
print("-" * 50)
print(f"Agent: {response}")
print("-" * 50)

### Context Isolation Explained

Notice how using `test_context_2` created a completely fresh conversation:

- **No Memory of Previous Booking**: The agent doesn't remember the New York to London booking
- **Fresh Start**: It treats this as a new conversation and asks for all required information
- **Independent State**: Each context maintains its own conversation history

This is crucial for multi-user scenarios where different users or sessions need isolated conversations.


---

### Understanding A2A Protocol Components

Now that we have a working flight booking agent, we need to make it available to other agents through the A2A protocol. This requires several components:

#### 1. **Agent Executor**
- Bridges between the A2A protocol and our agent
- Handles the conversion of A2A requests to agent method calls
- Manages the event queue for responses

#### 2. **Agent Card**
- Describes the agent's capabilities in a standardized format
- Lists available skills and their descriptions
- Provides examples of how to use the agent

#### 3. **A2A Server**
- Exposes the agent via HTTP endpoints
- Handles A2A protocol communication
- Manages requests and responses

Let's implement these components step by step, starting with the **Agent Executor**



In [ ]:
# Flight booking tool function for the Travel Agent
def book_flight(user_input: Annotated[str, Field(description="The user's flight booking request")]) -> str:
    """
    Book a flight using the external flight booking agent via A2A protocol.
    
    Args:
        user_input: The user's flight booking request
        
    Returns:
        The response from the flight booking agent
    """
    async def _book_flight_async():
        try:
            import httpx
            
            # A2A protocol communication setup
            flight_booking_agent_url = "http://localhost:8001"  # Assuming flight agent runs on port 8001
            
            async with httpx.AsyncClient() as httpx_client:
                resolver = A2ACardResolver(
                    httpx_client=httpx_client, 
                    base_url=flight_booking_agent_url
                )
                agent_card = await resolver.get_agent_card()
                
                client = A2AClient(httpx_client=httpx_client, agent_card=agent_card)
                
                request = SendMessageRequest(
                    id=str(uuid4()),
                    params=MessageSendParams(
                        message={
                            "messageId": uuid4().hex,
                            "role": "user",
                            "parts": [{"text": user_input}],
                            "contextId": "travel_booking_context",
                        }
                    )
                )
                
                response = await client.send_message(request)
                result = response.model_dump(mode='json', exclude_none=True)
                
                print(f"✓ A2A flight booking response received")
                return result["result"]["parts"][0]["text"]
                
        except Exception as e:
            print(f"✗ Error in A2A flight booking: {e}")
            return f"Sorry, I encountered an error while trying to book your flight: {str(e)}"
    
    # Run the async function
    return asyncio.run(_book_flight_async())

class TravelPlanningAgent:
    """A travel planning agent that can book flights through A2A communication."""
    
    def __init__(self):
        """Initialize the travel planning agent."""
        print("Initializing Travel Planning Agent...")
        
        # Create Azure client
        client = AzureChatClient(credential=AzureCliCredential())
        
        # Create agent with the book_flight tool - much simpler registration!
        self.agent = client.create_agent(
            instructions=(
                "You are a helpful travel planning assistant. "
                "Use the provided tools to assist users with their travel plans. "
                "When users ask about flights, use the book_flight tool to help them."
            ),
            tools=[book_flight]  # Direct function registration - no decorators needed!
        )
        
        print("✓ Travel Planning Agent initialized with flight booking capability!")
    
    async def chat(self, user_input: str) -> str:
        """
        Chat with the travel agent.
        
        Args:
            user_input: User's message
            
        Returns:
            Agent's response
        """
        try:
            # Simple run() method - unified API!
            response = await self.agent.run(user_input)
            return response.text
        except Exception as e:
            return f"Sorry, I encountered an error: {str(e)}"

# Create the travel agent
travel_agent = TravelPlanningAgent()
print("Travel agent ready!")

### Agent Executor Explained

The `SemanticKernelFlightBookingAgentExecutor` class serves as a crucial bridge:

1. **Protocol Translation**: It converts A2A protocol requests into calls to our agent's `chat` method
2. **Context Management**: It extracts context information from A2A requests
3. **Event Handling**: It manages the event queue to send responses back to the calling agent
4. **Error Handling**: It provides graceful error handling for various failure scenarios

The key methods are:
- `execute()`: Processes incoming A2A requests and calls our agent
- `cancel()`: Handles cancellation requests (not implemented for this example)


---


### Agent Card and Server Configuration

The configuration functions we just created serve important purposes:

#### **Agent Skill Definition**
- Defines what the agent can do (`flight_booking`)
- Provides human-readable descriptions
- Includes example usage patterns
- Uses tags for categorization

#### **Agent Card**
- Acts like a "business card" for the agent
- Specifies capabilities (like streaming support)
- Defines input/output modes (text in our case)
- Provides discovery information for other agents

#### **Server Application**
- Combines the executor, task store, and agent card
- Creates HTTP endpoints for A2A communication
- Handles the low-level protocol details



In [ ]:
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.utils import new_agent_text_message, new_task

class MicrosoftAgentFrameworkFlightBookingAgentExecutor(AgentExecutor):
    """Executor for Microsoft Agent Framework Flight Booking Agent that handles A2A protocol integration."""
    
    def __init__(self):
        """Initialize the executor with a flight booking agent instance."""
        print("Initializing Microsoft Agent Framework Flight Booking Agent Executor...")
        self.agent = MicrosoftAgentFrameworkFlightBookingAgent()
        print("✓ Microsoft Agent Framework Flight Booking Agent Executor initialized!")
    
    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        """
        Execute a flight booking request.
        
        Args:
            context: The request context containing user input and task information
            event_queue: Queue for sending events and responses
        """
        user_input = context.get_user_input()
        task = context.current_task
        context_id = context.context_id
        
        # Create a new task if one doesn't exist
        if not task:
            task = new_task(context.message)
            await event_queue.enqueue_event(task)
        
        print(f"Executing flight booking - User input: {user_input}, Task ID: {task.id}, Context ID: {context_id}")
        
        try:
            # Process the flight booking request using Microsoft Agent Framework
            result = await self.agent.book_flight(user_input, context_id)
            
            # Send the result back through the event queue
            await event_queue.enqueue_event(new_agent_text_message(result))
            
            print("✓ Flight booking executed successfully")
            
        except ValueError as ve:
            print(f"✗ Validation error during flight booking: {ve}")
            await event_queue.enqueue_event(
                new_agent_text_message(f"I need more information to help you book a flight: {str(ve)}")
            )
            
        except Exception as e:
            print(f"✗ Unexpected error during flight booking execution: {e}")
            await event_queue.enqueue_event(
                new_agent_text_message(
                    "I apologize, but I encountered an error while processing your flight booking request. "
                    "Please try again or contact support if the issue persists."
                )
            )
    
    async def cancel(
        self,
        context: RequestContext,
        event_queue: EventQueue
    ) -> None:
        """
        Handle cancellation requests.
        
        Args:
            context: The request context
            event_queue: Queue for sending events
            
        Raises:
            Exception: Cancel operation is not supported
        """
        print("⚠️ Cancel operation requested but not supported for flight booking agent")
        raise Exception('Cancel operation not supported for flight booking operations.')

# Test the executor
flight_executor = MicrosoftAgentFrameworkFlightBookingAgentExecutor()
print("Flight booking executor ready!")

### Starting the A2A Server

We're starting the server in a background thread so it doesn't block our notebook execution. The server will:

1. **Listen on port 9999** for incoming A2A requests
2. **Serve the agent card** at `/.well-known/agent.json` for discovery
3. **Handle A2A protocol messages** and route them to our agent
4. **Run as a daemon thread** so it stops when the notebook kernel stops

> **Note**: If you see an "address already in use" error, it means the server is already running from a previous execution.


In [ ]:
# Function to start the A2A server in a separate thread
def start_a2a_server():
    """Start the A2A server in a separate thread."""
    server = create_flight_booking_server()
    uvicorn.run(server, host=SERVER_HOST, port=SERVER_PORT)

# Start the server in a background thread
server_thread = threading.Thread(target=start_a2a_server, daemon=True)
server_thread.start()

print(f"✓ A2A server starting on {SERVER_HOST}:{SERVER_PORT}")
print("⏳ Waiting a moment for server to initialize...")

# Give the server a moment to start
import time
time.sleep(2)

In [ ]:
async with httpx.AsyncClient() as client:
    response = await client.get(f"http://localhost:{SERVER_PORT}/.well-known/agent.json")
    if response.status_code == 200:
        print("✅ A2A Server is running successfully!")
        agent_card = response.json()
        print(f"Agent Name: {agent_card.get('name')}")
        print(f"Agent Description: {agent_card.get('description')}")
        print(f"Agent Skills: {[skill.get('name') for skill in agent_card.get('skills', [])]}")

### Server Status Verification

Great! The server is running and responding to requests. The agent card endpoint (`/.well-known/agent.json`) is a standard A2A discovery mechanism that allows other agents to:

1. **Discover capabilities** - What skills the agent offers
2. **Understand interfaces** - How to communicate with the agent
3. **Get examples** - Sample requests to help with usage
4. **Check compatibility** - Protocol version and supported features

Now our flight booking agent is ready to be used by other agents through the A2A protocol!

---

### Travel Planning Agent Architecture

Now we create the orchestrating agent that will use our flight booking agent as a tool:

#### **Agent Configuration:**
- **General Purpose**: Unlike the specialized flight booking agent, this agent handles various travel tasks
- **Tool Integration**: Has access to the `FlightBookingTool` for flight-related requests
- **Smart Routing**: Decides when to use the flight booking tool vs. handling requests directly
- **Context Awareness**: Maintains its own conversation history separate from the flight booking agent

#### **How It Decides When to Use the Flight Booking Tool:**
The travel agent uses Semantic Kernel's function calling capabilities. When it detects flight-related requests in the conversation, it automatically calls the `book_flight` function, which triggers A2A communication with our flight booking agent.


In [ ]:
from a2a.server.endpoint import A2AServerEndpoint
from a2a.server.agent_card import AgentCard, AgentCapability, AgentSchema
import uvicorn
import asyncio
from fastapi import FastAPI

# Create the A2A Server for Flight Booking Agent
def create_flight_booking_server():
    """Create and configure the A2A server for flight booking agent."""
    
    app = FastAPI(
        title="Microsoft Agent Framework Flight Booking Agent",
        description="A flight booking agent using Microsoft Agent Framework with A2A protocol support"
    )
    
    # Create agent card
    agent_card = AgentCard(
        name='Microsoft Agent Framework Flight Booking Agent',
        description='An agent that helps users book flights using Microsoft Agent Framework capabilities.',
        publisher='Workshop Demo',
        version='1.0.0',
        capabilities=[
            AgentCapability(
                name='flight_booking',
                description='Book flights with comprehensive assistance',
                input_schema=AgentSchema(
                    type='object',
                    properties={
                        'message': {
                            'type': 'string',
                            'description': 'Flight booking request'
                        }
                    },
                    required=['message']
                )
            )
        ]
    )
    
    # Create A2A endpoint
    a2a_endpoint = A2AServerEndpoint(
        agent_card=agent_card,
        agent_executor=MicrosoftAgentFrameworkFlightBookingAgentExecutor(),
    )
    
    # Mount the A2A endpoint
    app.mount('/a2a', a2a_endpoint)
    
    return app

# Create the server app
flight_booking_app = create_flight_booking_server()
print("✓ Flight booking A2A server created!")
print("  - Name: Microsoft Agent Framework Flight Booking Agent")
print("  - Capability: flight_booking")
print("  - Ready to receive A2A requests at /a2a endpoint")

In [ ]:
# Import required modules for the server
import threading
import httpx

SERVER_HOST = "localhost"
SERVER_PORT = 9999

# Chat function for testing
async def chat_with_travel_agent(user_input: str, context_id: str) -> str:
    """Helper function to chat with the travel agent."""
    try:
        # Create Azure client
        client = AzureChatClient(credential=AzureCliCredential())
        
        # Define travel agent (simplified for testing)
        travel_agent = client.create_agent(
            instructions=(
                "You are a helpful travel planning assistant. "
                "Use the provided tools to assist users with their travel plans. "
                "When users ask about flights, use the book_flight_tool to help them."
            ),
            tools=[book_flight_tool]
        )
        
        # Use the agent
        response = await travel_agent.run(user_input)
        return response.text
        
    except Exception as e:
        return f"Sorry, I encountered an error: {str(e)}"

# Travel Agent Tool Function (updated for Microsoft Agent Framework)
def book_flight_tool(user_input: Annotated[str, Field(description="The user's flight booking request")]) -> str:
    """
    Book a flight using the external flight booking agent via A2A protocol.
    
    Args:
        user_input: The user's flight booking request
        
    Returns:
        The response from the flight booking agent
    """
    async def _book_flight_async():
        try:
            import httpx
            
            # A2A protocol communication setup
            flight_booking_agent_url = f"http://{SERVER_HOST}:{SERVER_PORT}"  # Flight agent server URL
            
            async with httpx.AsyncClient() as httpx_client:
                resolver = A2ACardResolver(
                    httpx_client=httpx_client, 
                    base_url=flight_booking_agent_url
                )
                agent_card = await resolver.get_agent_card()
                
                client = A2AClient(httpx_client=httpx_client, agent_card=agent_card)
                
                request = SendMessageRequest(
                    id=str(uuid4()),
                    params=MessageSendParams(
                        message={
                            "messageId": uuid4().hex,
                            "role": "user",
                            "parts": [{"text": user_input}],
                            "contextId": "travel_booking_context",
                        }
                    )
                )
                
                response = await client.send_message(request)
                result = response.model_dump(mode='json', exclude_none=True)
                
                print(f"✓ A2A response: {result['result']['parts'][0]['text'][:100]}...")
                return result["result"]["parts"][0]["text"]
                
        except Exception as e:
            error_msg = f"Sorry, I encountered an error while trying to book your flight: {str(e)}"
            print(f"✗ A2A Error: {error_msg}")
            return error_msg
    
    # Run the async function synchronously
    return asyncio.run(_book_flight_async())

# Create Travel Booking Agent with Web Interface
from fastapi import FastAPI, Request, Form
from fastapi.responses import HTMLResponse

def create_travel_booking_server():
    """Create the travel booking agent server with web interface."""
    
    app = FastAPI(
        title="Travel Booking Agent", 
        description="A travel planning assistant with flight booking capabilities using Microsoft Agent Framework"
    )
    
    # Create the travel agent with Microsoft Agent Framework
    client = AzureChatClient(credential=AzureCliCredential())
    travel_agent = client.create_agent(
        instructions=(
            "You are a helpful travel planning assistant. "
            "Use the provided tools to assist users with their travel plans. "
            "When users ask about flights, use the book_flight_tool to help them."
        ),
        tools=[book_flight_tool]  # Simple tool registration!
    )
    
    # Store chat threads for different users
    chat_threads = {}
    
    @app.post("/chat")
    async def chat(user_input: str = Form(...), context_id: str = Form("default")):
        """Handle chat requests from users."""
        print(f"Chat request: {user_input} (Context: {context_id})")
        
        try:
            # Get or create thread for this context
            if context_id not in chat_threads:
                chat_threads[context_id] = travel_agent.get_new_thread()
            
            thread = chat_threads[context_id]
            
            # Use Microsoft Agent Framework's unified run() API
            response = await travel_agent.run(user_input, thread=thread)
            
            print(f"✓ Travel agent response generated")
            return {"response": response.text}
            
        except Exception as e:
            print(f"✗ Error processing chat request: {e}")
            return {"response": "Sorry, I encountered an error processing your request. Please try again."}
    
    @app.get("/", response_class=HTMLResponse)
    async def index():
        """Serve the main HTML interface."""
        return HTMLResponse(content="""
        <!DOCTYPE html>
        <html>
        <head>
            <title>Travel Booking Agent - Microsoft Agent Framework</title>
        </head>
        <body>
            <h1>Travel Booking Agent</h1>
            <p>Powered by Microsoft Agent Framework</p>
            <form action="/chat" method="post">
                <input type="text" name="user_input" placeholder="Ask me about travel plans..." required>
                <input type="hidden" name="context_id" value="web_user">
                <button type="submit">Send</button>
            </form>
        </body>
        </html>
        """)
    
    return app

# Create the travel booking server
travel_booking_app = create_travel_booking_server()
print("✓ Travel booking server created with Microsoft Agent Framework!")
print("  - Web interface available at /")
print("  - Chat API available at /chat")
print("  - Uses book_flight_tool for A2A communication")

### Test the A2A Communication

Now let's test the Agent-to-Agent communication! We'll chat with the Travel Planning Agent, which will use the Flight Booking Agent when needed:


In [ ]:
# Test 1: General travel question (should not trigger flight booking)
print("🧪 Test 1: General travel question")
response = await chat_with_travel_agent(
    "What are some good travel destinations for summer?",
    "demo_context"
)
print("-" * 50)
print("Final Response")
print("-" * 50)
print(f"Agent: {response}")
print("-" * 50)

In [ ]:
# Test 2: Flight booking request (should trigger A2A communication)
print("🧪 Test 2: Flight booking request (A2A communication)")
response = await chat_with_travel_agent(
    "I need to book a flight from San Francisco to Tokyo on December 25th",
    "demo_context"
)
print("-" * 50)
print("Final Response")
print("-" * 50)
print(f"Agent: {response}")
print("-" * 50)

In [ ]:

    # Test 3: Follow-up with more flight details (should continue A2A communication)
print("🧪 Test 3: Follow-up with more flight details")
response = await chat_with_travel_agent(
    "Make it for 2 passengers, and I prefer business class",
    "demo_context"
)
print("-" * 50)
print("Final Response")
print("-" * 50)
print(f"Agent: {response}")
print("-" * 50)

### Understanding the Test Results

Let's analyze what happened in our A2A communication tests:

#### **Test 1: General Travel Question**
- **No A2A Call**: The travel agent handled this directly without calling the flight booking agent
- **Direct Response**: It provided travel destination recommendations from its own knowledge
- **Smart Routing**: It correctly identified this as a general travel question, not a flight booking request

#### **Test 2: Flight Booking Request**
- **A2A Triggered**: The travel agent detected flight booking intent and called the flight booking tool
- **Protocol Communication**: We can see the HTTP requests to the flight booking agent in the logs
- **Agent Card Fetching**: The system fetched the agent card to understand capabilities
- **Message Routing**: The request was properly formatted and sent to the flight booking agent
- **Response Handling**: The flight booking agent's response was returned through the travel agent

#### **Test 3: Follow-up Details**
- **Continued A2A**: The travel agent again used A2A to send additional details
- **Context Preservation**: The flight booking agent maintained context from the previous interaction
- **Successful Completion**: The booking was completed with all necessary information

This demonstrates the power of A2A communication - agents can specialize while working together seamlessly!


In [ ]:
print("🧪 Test 4: Mixed request (travel advice + flight booking)")
response = await chat_with_travel_agent(
    "What's the best time to visit Paris? Also, can you book me a flight there for next month?",
    "demo_context2"
)
print("-" * 50)
print("Final Response")
print("-" * 50)
print(f"Agent: {response}")
print("-" * 50)

In [ ]:
print("🧪 Test 5: Follow up to finalize the booking)")
response = await chat_with_travel_agent(
    "It's going to be just for myself, departing from Cairo on the 15th of November for 5 days and lets do Economy class",
    "demo_context2"
)
print("-" * 50)
print("Final Response")
print("-" * 50)
print(f"Agent: {response}")
print("-" * 50)

## Summary

Congratulations! You've successfully implemented Agent-to-Agent (A2A) communication using Microsoft Agent Framework. Here's what we accomplished:

### Key Achievements

1. **Microsoft Agent Framework Integration**: Created agents using the new simplified framework
2. **Tool Registration**: Implemented direct function registration without decorators
3. **A2A Communication**: Established inter-agent communication protocol
4. **Production Deployment**: Built FastAPI servers for both agents
5. **Automatic Thread Management**: Leveraged built-in conversation state handling

### Microsoft Agent Framework Benefits Demonstrated

- **Simplified API**: Single `run()` method replaced complex invocation patterns
- **Direct Client Usage**: No kernel dependency or complex setup required
- **Built-in Thread Management**: Automatic conversation state handling
- **Easy Tool Registration**: Direct function registration with type hints
- **Azure Integration**: Native Azure CLI credential support

### Architecture Overview

- **Travel Agent**: Uses Microsoft Agent Framework with book_flight tool for A2A calls
- **Flight Agent**: Microsoft Agent Framework agent exposed via A2A protocol  
- **Communication**: HTTP-based A2A protocol enables seamless agent collaboration
- **State Management**: Each agent maintains its own conversation threads
- **Production Ready**: FastAPI servers with proper error handling and logging

### Key Improvements over Previous Approaches

- Microsoft Agent Framework's function calling mechanism works seamlessly with A2A
- Simplified imports and reduced boilerplate code
- Better error handling and resource management
- More intuitive API design with unified `run()` method
- Built-in authentication and Azure integration

### Interactive Chat

You can now interact with the Travel Planning Agent directly. Try asking about travel destinations, flight bookings, or any travel-related questions:


In [ ]:
# Interactive Chat Function
async def interactive_chat():
    """Interactive chat with the Travel Planning Agent."""
    print("💬 Starting interactive chat with Travel Planning Agent")
    print("Type 'exit' to end the conversation")
    print("-" * 50)
    
    context_id = "interactive_session"
    
    while True:
        try:
            # In a real Jupyter environment, you might want to use input() or ipywidgets
            user_input = input("You: ")
            
            if user_input.lower() in ['exit', 'quit', 'bye']:
                print("👋 Thank you for chatting! Goodbye!")
                break
                
            if user_input.strip():
                response = await chat_with_travel_agent(user_input, context_id)
                print(f"Travel Agent: {response}")
                print("-" * 50)
                
        except KeyboardInterrupt:
            print("\n👋 Chat ended by user. Goodbye!")
            break
        except Exception as e:
            print(f"❌ Error: {e}")
            print("-" * 50)

# Example usage (uncomment to run interactively)
# await interactive_chat()

# For demonstration, let's show some example interactions
print("💡 Example interactions you can try:")
print("- 'What are the best beaches in Thailand?'")
print("- 'Book a flight from London to New York tomorrow'")
print("- 'What's the weather like in Tokyo in spring?'")
print("- 'I need a round-trip flight to Paris for 2 people'")
print("- 'Tell me about travel insurance options'")
print("")
print("💬 To start interactive chat, uncomment and run: await interactive_chat()")


### A2A Communication Flow Diagram

Here's what happens when the travel agent needs to book a flight:

```
┌─────────────────┐    1. User Request   ┌─────────────────┐
│                 │ ───────────────────► │                 │
│      User       │                      │ Travel Planning │
│                 │ ◄─────────────────── │     Agent       │
└─────────────────┘    6. Final Response └─────────────────┘
                                                   │
                                                   │ 2. Detects Flight
                                                   │    Booking Intent
                                                   ▼
                                         ┌─────────────────┐
                                         │                 │
                                         │  FlightBooking  │
                                         │     Tool        │
                                         │                 │
                                         └─────────────────┘
                                                   │
                                                   │ 3. A2A Protocol
                                                   │    Communication
                                                   ▼
                                         ┌─────────────────┐
                                         │                 │
                                         │ Flight Booking  │
                                         │     Agent       │
                                         │   (A2A Server)  │
                                         │                 │
                                         └─────────────────┘
                                                   │
                                                   │ 4. Processes Request
                                                   │ 5. Returns Response
                                                   ▼
```

This architecture allows for:
- **Separation of Concerns**: Each agent has a specific responsibility
- **Reusability**: The flight booking agent can be used by multiple travel agents
- **Scalability**: Agents can be deployed and scaled independently
- **Maintainability**: Changes to flight booking logic don't affect travel planning logic



## Summary and Key Concepts

Congratulations! You've successfully implemented Agent-to-Agent (A2A) communication using Semantic Kernel. Here's what we accomplished:

### 🎯 What We Built

1. **Flight Booking Agent**: A specialized agent that handles flight bookings
2. **A2A Server**: Exposes the flight booking agent via HTTP API
3. **Travel Planning Agent**: A general travel agent that uses the flight booking agent as a tool
4. **A2A Communication**: Seamless communication between agents

### 🔑 Key Concepts Learned

#### 1. **Agent Specialization**
- Each agent has a specific purpose and set of capabilities
- Specialization leads to better performance and maintainability
- Agents can be developed and deployed independently

#### 2. **A2A Protocol**
- Standardized way for agents to communicate
- Includes agent cards, skill definitions, and message formats
- Enables discovery and invocation of agent capabilities

#### 3. **Tool Integration**
- Agents can use other agents as tools
- Semantic Kernel's function calling mechanism works with A2A
- Allows for complex workflows and orchestration

#### 4. **Context Management**
- Each agent maintains its own conversation context
- Context IDs ensure proper conversation tracking
- Enables stateful interactions across agent boundaries

### 🏗️ Architecture Benefits

1. **Modularity**: Each agent can be developed, tested, and deployed separately
2. **Scalability**: Agents can be scaled independently based on demand
3. **Reusability**: Specialized agents can be used by multiple orchestrators
4. **Maintainability**: Clear separation of concerns makes code easier to maintain

### 🚀 Next Steps

To extend this implementation, you could:

1. **Add More Specialized Agents**: Hotel booking, car rental, restaurant recommendations
2. **Add More Tools**: Integrate with real booking APIs and services

This tutorial demonstrates the power of agent-to-agent communication in creating sophisticated, distributed AI systems. The modular approach allows for building complex workflows while maintaining clean, maintainable code."


## Optional: Cleanup

If you want to stop the A2A server that's running in the background, you can run the following code:


In [ ]:
# Optional: Cleanup
# Note: The server thread is running as a daemon thread, so it will automatically
# terminate when the main Python process ends. If you want to explicitly check
# if the server is still running, you can use:

if server_thread.is_alive():
    print("✅ A2A Server is still running in the background")
    print("🔧 The server will automatically stop when the notebook kernel is restarted")
else:
    print("❌ A2A Server thread has stopped")

# You can also check the server status again
print("\n🔍 Final server status check:")
try:
    async with httpx.AsyncClient() as client:
        response = await client.get(f"http://localhost:{SERVER_PORT}/.well-known/agent.json")
        if response.status_code == 200:
            print("✅ A2A Server is still responding")
        else:
            print(f"❌ A2A Server returned status code: {response.status_code}")
except Exception as e:
    print(f"❌ Cannot connect to A2A Server: {e}")

print("\n🎉 Tutorial completed successfully!")
print("💡 Remember: The A2A server runs as a daemon thread and will stop automatically when the notebook kernel is restarted.")


## Running the Standalone Agent Files

This tutorial includes standalone implementations of both agents that can be run independently:

### Flight Booking Agent Server
The flight booking agent is available as a standalone server in the `flight-booking-agent/` directory:

```bash
cd flight-booking-agent/
uv run server.py
```

This will start the A2A server on `http://localhost:9999` with the flight booking agent.

### Travel Booking Agent
The travel booking agent with a web interface is available in the `travel-booking-agent/` directory:

```bash
cd travel-booking-agent/
uv run agent.py
```

This will start a local web server where you can interact with the travel planning agent through a browser interface.